# Phase 1: PINN Basin Maps on GPU

2D basin map (seed × learning_rate) for 1D convection PINN.

**Before running:** Runtime → Change runtime type → **A100 GPU**

In [ ]:
# 1. Setup
!git clone https://github.com/dzmitrybudzko/math-ai.git
import sys
sys.path.insert(0, 'math-ai/experiments/phase1')

import torch
print(f'PyTorch {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# 2. Quick sanity check — single PINN run on GPU
from convection_pinn import train_single, CORRECT, TRIVIAL, DIVERGED, PINN
import time

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

# Warmup CUDA (first run is always slow)
_ = torch.randn(100, 100, device=device) @ torch.randn(100, 100, device=device)

# Use larger network (4x50) to better utilize GPU
layers_gpu = [2, 50, 50, 50, 50, 1]

t0 = time.time()
outcome, l2, loss = train_single(seed=0, beta=1.0, method='adam', n_adam=2000, device=device)
dt = time.time() - t0

labels = {CORRECT: 'CORRECT', TRIVIAL: 'TRIVIAL', DIVERGED: 'DIVERGED'}
print(f'Run 1 (with CUDA init): {dt:.1f}s')

# Second run — true timing
t0 = time.time()
outcome, l2, loss = train_single(seed=1, beta=1.0, method='adam', n_adam=2000, device=device)
dt = time.time() - t0
print(f'Run 2 (steady state):   {dt:.1f}s')
print(f'Outcome: {labels[outcome]}, L2={l2:.4f}, loss={loss:.2e}')
print(f'\nAt {dt:.0f}s/run, 900 runs (30x30 grid) = ~{900*dt/60:.0f} min')

In [ ]:
# 3. 2D Basin Map: beta=1 (sweet spot), 30 seeds x 30 learning rates
import numpy as np
from convection_pinn import compute_basin_map_2d, plot_basin_2d

beta = 1.0
n_seeds = 30
lr_values = np.logspace(-5, -1, 30)

print(f'Basin map: beta={beta}, {n_seeds} seeds x {len(lr_values)} LRs = {n_seeds * len(lr_values)} runs')

t0 = time.time()
outcomes, l2_errors, lr_vals = compute_basin_map_2d(
    beta=beta, method='adam', n_seeds=n_seeds,
    lr_values=lr_values, n_adam=2000, device=device
)
dt = time.time() - t0

n_correct = np.sum(outcomes == CORRECT)
n_trivial = np.sum(outcomes == TRIVIAL)
n_diverged = np.sum(outcomes == DIVERGED)
total = outcomes.size

print(f'\nDone in {dt:.0f}s ({dt/60:.1f} min)')
print(f'Correct: {n_correct}/{total} ({100*n_correct/total:.1f}%)')
print(f'Trivial: {n_trivial}/{total} ({100*n_trivial/total:.1f}%)')
print(f'Diverged: {n_diverged}/{total} ({100*n_diverged/total:.1f}%)')

In [ ]:
# 4. Visualize basin map
import matplotlib.pyplot as plt

plot_basin_2d(outcomes, lr_vals,
              title=f'2D Basin Map: Adam, beta={beta}, 2000 steps',
              filename=f'basin_2d_adam_beta{beta}.png')

# Also display inline
colors_map = np.array([[0.0, 0.8, 0.0], [1.0, 0.6, 0.0], [0.2, 0.2, 0.2]])
img = colors_map[outcomes]

fig, ax = plt.subplots(figsize=(10, 6))
ax.imshow(img, aspect='auto', origin='lower',
          extent=[np.log10(lr_vals[0]), np.log10(lr_vals[-1]), 0, n_seeds])
ax.set_xlabel('log10(learning rate)')
ax.set_ylabel('Seed')
ax.set_title(f'2D Basin Map: Adam, beta={beta}, 2000 steps')

from matplotlib.patches import Patch
legend = [Patch(facecolor=colors_map[0], label=f'Correct ({100*n_correct/total:.0f}%)'),
          Patch(facecolor=colors_map[1], label=f'Trivial ({100*n_trivial/total:.0f}%)'),
          Patch(facecolor=colors_map[2], label=f'Diverged ({100*n_diverged/total:.0f}%)')]
ax.legend(handles=legend, loc='upper right')
plt.show()

In [ ]:
# 5. Basin entropy (Daza et al. 2016)
def basin_entropy(outcomes, box_size=5):
    """Compute basin entropy S_b for a 2D outcome grid."""
    h, w = outcomes.shape
    entropies = []
    for i in range(0, h - box_size + 1):
        for j in range(0, w - box_size + 1):
            box = outcomes[i:i+box_size, j:j+box_size].ravel()
            unique, counts = np.unique(box, return_counts=True)
            probs = counts / counts.sum()
            entropies.append(-np.sum(probs * np.log(probs)))
    return np.mean(entropies)

S_b = basin_entropy(outcomes)
print(f'Basin entropy S_b = {S_b:.4f}')
print(f'S_b = 0 means all boxes are single-outcome (clean basins)')
print(f'S_b > 0 means mixed boundaries (fractal-like structure)')

In [ ]:
# 6. Save results for later analysis
np.savez('basin_2d_adam_beta1.npz',
         outcomes=outcomes, l2_errors=l2_errors,
         lr_values=lr_vals, beta=beta,
         basin_entropy=S_b)
print('Saved: basin_2d_adam_beta1.npz')

## Next: More beta values

Run the same map for beta=5, 10, 30 to see how basin structure changes with difficulty.

In [ ]:
# 7. Multi-beta sweep (optional — run if time/units allow)
betas = [5.0, 10.0, 30.0]
results = {}

for beta in betas:
    print(f'\n{"="*60}')
    print(f'Beta = {beta}')
    print(f'{"="*60}')
    
    t0 = time.time()
    out, l2e, lrs = compute_basin_map_2d(
        beta=beta, method='adam', n_seeds=50,
        lr_values=np.logspace(-5, -1, 50), n_adam=2000, device=device
    )
    dt = time.time() - t0
    
    S_b = basin_entropy(out)
    n_c = np.sum(out == CORRECT)
    total = out.size
    
    print(f'Done in {dt:.0f}s. Correct: {n_c}/{total} ({100*n_c/total:.1f}%). S_b={S_b:.4f}')
    
    plot_basin_2d(out, lrs,
                  title=f'2D Basin Map: Adam, beta={beta}, 2000 steps (S_b={S_b:.3f})',
                  filename=f'basin_2d_adam_beta{int(beta)}.png')
    
    results[beta] = {'outcomes': out, 'l2_errors': l2e, 'entropy': S_b}
    
    np.savez(f'basin_2d_adam_beta{int(beta)}.npz',
             outcomes=out, l2_errors=l2e, lr_values=lrs,
             beta=beta, basin_entropy=S_b)

print('\n\nSummary:')
print(f'{"beta":>6} | {"correct%":>8} | {"S_b":>6}')
print('-' * 30)
for b, r in results.items():
    nc = np.sum(r['outcomes'] == CORRECT)
    print(f'{b:6.0f} | {100*nc/r["outcomes"].size:7.1f}% | {r["entropy"]:.4f}')

In [ ]:
# 8. Download results
from google.colab import files
import glob

for f in glob.glob('basin_2d_*.npz') + glob.glob('basin_2d_*.png'):
    print(f'Downloading: {f}')
    files.download(f)